# Data Quality Tutorial and Lab

## Understanding Data Quality Dimensions

Before coding, let's understand what we're measuring:
1. Accuracy

Does the data correctly represent real-world values?

    Example: Height of 15 cm for an adult is inaccurate

2. Completeness

Are all required fields populated?

    Example: Missing age values

3. Uniqueness

Are there duplicate records?

    Example: Same person appearing twice

4. Consistency

Is data uniform across the dataset?

    Example: Age should match Date of Birth

5. Timeliness

Is the data up-to-date?

    Example: Records not updated in 2 years

6. Validity

Does data conform to expected formats/rules?

    Example: Email should contain "@"


## Lab 1: Basic Data Quality Checks with pandas_dq

In [1]:
!pip install pandas_dq

### Step 1: Import Libraries and Create Sample Data

In [2]:
import pandas as pd
import numpy as np
from pandas_dq import dq_report, DataSchemaChecker

# Create a sample dataset with quality issues
data = {
    "Name": ["John Doe", "Jane Smith", "Johnathan Doe", "Jake Brown", "Jake Brown"],
    "Email": [
        "john@example.com",
        "jane@example,com",  # Invalid: comma instead of dot
        "johnathan.example.com",  # Invalid: missing @
        "jake.brown@example.com",
        "jake.brown@example.com"  # Duplicate
    ],
    "EyeColor": ["Blue", "Green", "Blue", "Pale", "Blue"],  # "Pale" is unusual
    "Age": [30, None, 50, 45, 45],  # Missing value
    "Height": [173, 160, 15, 180, 180],  # 15 cm is unrealistic
    "DateOfBirth": [
        "1989-07-11",
        "1992-03-05",
        "1970-07-11",
        "1977-02-10",
        "1977-02-10"
    ],
    "Address": [
        "123 Fake St",
        "456 Real Rd",
        "123 Fake St",
        "789 Another Pl",
        "789 Another Pl"
    ],
    "LastUpdated": [
        "2025-01-01",
        "2024-12-31",
        "2022-05-25",  # Old record
        "2023-10-01",  # Old record
        "2023-10-01"
    ]
}

df = pd.DataFrame(data)
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (5, 8)


,Name,Email,EyeColor,Age,Height,DateOfBirth,Address,LastUpdated
0,John Doe,john@example.com,Blue,30.0,173,1989-07-11,123 Fake St,2025-01-01
1,Jane Smith,"jane@example,com",Green,NaN,160,1992-03-05,456 Real Rd,2024-12-31
2,Johnathan Doe,johnathan.example.com,Blue,50.0,15,1970-07-11,123 Fake St,2022-05-25
3,Jake Brown,jake.brown@example.com,Pale,45.0,180,1977-02-10,789 Another Pl,2023-10-01
4,Jake Brown,jake.brown@example.com,Blue,45.0,180,1977-02-10,789 Another Pl,2023-10-01


### Step 2: Generate Automated Data Quality Report

In [4]:
# Generate comprehensive DQ report
report = dq_report(df, target=None, verbose=1)
# report

    All variables classified into correct types.


/tmp/ipykernel_1615/283899804.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  report = dq_report(df, target=None, verbose=1)


,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
Name,object,0.000000,80,,,No issue
Email,object,0.000000,80,,,No issue
EyeColor,object,0.000000,60,,,No issue
Age,float64,20.000000,NA,30.000000,50.000000,"1 missing values. Impute them with mean, median, mode, or a constant value such as 123., Column has 1 outliers greater than upper bound (53.75) or lower than lower bound(33.75). Cap them or remove them."
Height,int64,0.000000,80,15.000000,180.000000,Column has 1 outliers greater than upper bound (210.00) or lower than lower bound(130.00). Cap them or remove them.
DateOfBirth,object,0.000000,80,,,No issue
Address,object,0.000000,60,,,No issue
LastUpdated,object,0.000000,80,,,No issue


What this shows:

    Missing values percentage
    Unique values percentage
    Min/Max values
    Outliers detected
    Data type issues


### Step 3: Check Data Schema

In [5]:
# Define expected schema
schema = {
    'Name': 'object',
    'Email': 'object',
    'EyeColor': 'object',
    'Age': 'float64',
    'Height': 'float64',
    'DateOfBirth': 'object',
    'Address': 'object',
    'LastUpdated': 'object'
}

# Create schema checker
checker = DataSchemaChecker(schema)

# Fit the checker
checker.fit(df)

# Transform to fix issues
df_fixed = checker.transform(df)

print("Fixed DataFrame Info:")
df_fixed.info()

,column,expected_dtype,actual_dtype,data_dtype_mismatch
0,Height,float64,int64,Column 'Height' has data type 'int64' but expected 'float64'


Fixed DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Name         5 non-null      object 
 1   Email        5 non-null      object 
 2   EyeColor     5 non-null      object 
 3   Age          4 non-null      float64
 4   Height       5 non-null      float64
 5   DateOfBirth  5 non-null      object 
 6   Address      5 non-null      object 
 7   LastUpdated  5 non-null      object 
dtypes: float64(2), object(6)
memory usage: 452.0+ bytes


## Lab 2: Computing Data Quality Dimensions Manually

### Step 1: Setup

In [7]:
import pandas as pd
import numpy as np
from datetime import datetime
import re

# Use the same dataset from Lab 1
df = pd.DataFrame(data)
df.head()

,Name,Email,EyeColor,Age,Height,DateOfBirth,Address,LastUpdated
0,John Doe,john@example.com,Blue,30.0,173,1989-07-11,123 Fake St,2025-01-01
1,Jane Smith,"jane@example,com",Green,NaN,160,1992-03-05,456 Real Rd,2024-12-31
2,Johnathan Doe,johnathan.example.com,Blue,50.0,15,1970-07-11,123 Fake St,2022-05-25
3,Jake Brown,jake.brown@example.com,Pale,45.0,180,1977-02-10,789 Another Pl,2023-10-01
4,Jake Brown,jake.brown@example.com,Blue,45.0,180,1977-02-10,789 Another Pl,2023-10-01


### Step 2: Check Completeness
Check for missing values in each column

In [8]:
def check_completeness(df):
    """Check for missing values in each column"""
    missing_counts = df.isnull().sum()
    total_rows = len(df)
    missing_percentage = (missing_counts / total_rows) * 100

    completeness_df = pd.DataFrame({
        'Column': missing_counts.index,
        'Missing_Count': missing_counts.values,
        'Missing_Percentage': missing_percentage.values,
        'Complete_Percentage': 100 - missing_percentage.values
    })

    return completeness_df

# Run completeness check
completeness_report = check_completeness(df)
print("=== COMPLETENESS REPORT ===")
print(completeness_report)

=== COMPLETENESS REPORT ===
     Column     Missing_Count  Missing_Percentage  Complete_Percentage
0         Name        0                0.0                100.0       
1        Email        0                0.0                100.0       
2     EyeColor        0                0.0                100.0       
3          Age        1               20.0                 80.0       
4       Height        0                0.0                100.0       
5  DateOfBirth        0                0.0                100.0       
6      Address        0                0.0                100.0       
7  LastUpdated        0                0.0                100.0       


### Step 3: Check Uniqueness (Duplicates)
Check for duplicate rows

In [9]:
def check_uniqueness(df, subset=None):
    """Check for duplicate rows"""
    if subset is None:
        subset = df.columns.tolist()

    total_rows = len(df)
    duplicate_flags = df.duplicated(subset=subset, keep=False)
    duplicates_count = duplicate_flags.sum()

    print("=== UNIQUENESS REPORT ===")
    print(f"Total rows: {total_rows}")
    print(f"Duplicate rows: {duplicates_count}")
    print(f"Duplicate percentage: {(duplicates_count/total_rows)*100:.2f}%")
    print(f"\nDuplicate records:")
    print(df[duplicate_flags])

    return {
        'total_rows': total_rows,
        'duplicate_count': duplicates_count,
        'duplicate_percentage': round((duplicates_count/total_rows)*100, 2)
    }

# Check for duplicates based on Name and DateOfBirth
uniqueness_report = check_uniqueness(df, subset=['Name', 'DateOfBirth'])

=== UNIQUENESS REPORT ===
Total rows: 5
Duplicate rows: 2
Duplicate percentage: 40.00%

Duplicate records:
      Name             Email          EyeColor   Age  Height DateOfBirth  \
3  Jake Brown  jake.brown@example.com   Pale    45.0    180   1977-02-10   
4  Jake Brown  jake.brown@example.com   Blue    45.0    180   1977-02-10   

      Address     LastUpdated  
3  789 Another Pl  2023-10-01  
4  789 Another Pl  2023-10-01  


### Step 4: Check Validity
Check if data conforms to expected formats
- Email validity (must contain @)
- Check eye color validity must be one of (Blue, Green, Brown, Hazel, Gray)

In [10]:
def check_validity(df):
    """Check if data conforms to expected formats"""
    validity_results = {}

    # Email validity (must contain @)
    df['ValidEmail'] = df['Email'].apply(lambda x: '@' in str(x))
    validity_results['Email'] = {
        'valid_count': df['ValidEmail'].sum(),
        'invalid_count': (~df['ValidEmail']).sum(),
        'valid_percentage': (df['ValidEmail'].mean() * 100)
    }

    # Eye color validity (common colors)
    valid_colors = ['Blue', 'Green', 'Brown', 'Hazel', 'Gray']
    df['ValidEyeColor'] = df['EyeColor'].isin(valid_colors)
    validity_results['EyeColor'] = {
        'valid_count': df['ValidEyeColor'].sum(),
        'invalid_count': (~df['ValidEyeColor']).sum(),
        'valid_percentage': (df['ValidEyeColor'].mean() * 100)
    }

    print("=== VALIDITY REPORT ===")
    for column, metrics in validity_results.items():
        print(f"\n{column}:")
        print(f"  Valid: {metrics['valid_count']} ({metrics['valid_percentage']:.1f}%)")
        print(f"  Invalid: {metrics['invalid_count']} ({100-metrics['valid_percentage']:.1f}%)")

    return validity_results

validity_report = check_validity(df)

=== VALIDITY REPORT ===

Email:
  Valid: 4 (80.0%)
  Invalid: 1 (20.0%)

EyeColor:
  Valid: 4 (80.0%)
  Invalid: 1 (20.0%)


### Step 5: Check Accuracy
Check if values are realistic/plausible
- Height accuracy (50-250 cm for humans)
- Age accuracy (0-120 years)

In [13]:
def check_accuracy(df):
    """Check if values are realistic/plausible"""
    accuracy_issues = {}

    # Height accuracy (50-250 cm for humans)
    height_issues = df[(df['Height'] < 50) | (df['Height'] > 250)]
    accuracy_issues['Height'] = {
        'issue_count': len(height_issues),
        'issue_percentage': (len(height_issues)/len(df))*100,
        'valid_range': '50-250 cm',
        'problematic_records': height_issues.index.tolist()
    }

    # Age accuracy (0-120 years)
    age_issues = df[(df['Age'] < 0) | (df['Age'] > 120)]
    accuracy_issues['Age'] = {
        'issue_count': len(age_issues),
        'issue_percentage': (len(age_issues)/len(df))*100,
        'valid_range': '0-120 years'
    }

    print("=== ACCURACY REPORT ===")
    for column, metrics in accuracy_issues.items():
        print(f"\n{column}:")
        print(f"  Issues found: {metrics['issue_count']} ({metrics['issue_percentage']:.1f}%)")
        print(f"  Valid range: {metrics['valid_range']}")

    return accuracy_issues

accuracy_report = check_accuracy(df)

=== ACCURACY REPORT ===

Height:
  Issues found: 1 (20.0%)
  Valid range: 50-250 cm

Age:
  Issues found: 0 (0.0%)
  Valid range: 0-120 years


### Step 6: Check Consistency
Check if Age matches DateOfBirth

In [12]:
def check_consistency_age_dob(df, reference_date=None):
    """Check if Age matches DateOfBirth"""
    if reference_date is None:
        reference_date = datetime(2025, 3, 25)

    # Convert DateOfBirth to datetime
    df['DateOfBirth'] = pd.to_datetime(df['DateOfBirth'], errors='coerce')

    # Calculate age from DOB
    def calculate_age(dob):
        if pd.isnull(dob):
            return np.nan
        years = reference_date.year - dob.year
        if (reference_date.month, reference_date.day) < (dob.month, dob.day):
            years -= 1
        return years

    df['CalculatedAge'] = df['DateOfBirth'].apply(calculate_age)
    df['AgeDifference'] = abs(df['CalculatedAge'] - df['Age'].fillna(-9999))

    # Flag inconsistencies (difference > 1 year)
    inconsistent = df[df['AgeDifference'] > 1]

    print("=== CONSISTENCY REPORT (Age vs DOB) ===")
    print(f"Inconsistent records: {len(inconsistent)}")
    print(f"\nInconsistent records details:")
    print(inconsistent[['Name', 'Age', 'DateOfBirth', 'CalculatedAge', 'AgeDifference']])

    return {
        'inconsistent_count': len(inconsistent),
        'inconsistent_percentage': (len(inconsistent)/len(df))*100,
        'inconsistent_indices': inconsistent.index.tolist()
    }

consistency_report = check_consistency_age_dob(df)

=== CONSISTENCY REPORT (Age vs DOB) ===
Inconsistent records: 5

Inconsistent records details:
       Name        Age DateOfBirth  CalculatedAge  AgeDifference
0       John Doe  30.0  1989-07-11       35               5.0   
1     Jane Smith   NaN  1992-03-05       33           10032.0   
2  Johnathan Doe  50.0  1970-07-11       54               4.0   
3     Jake Brown  45.0  1977-02-10       48               3.0   
4     Jake Brown  45.0  1977-02-10       48               3.0   


### Step 7: Check Timeliness
Check if data is up-to-date (example data updated before a specific date 25/03/2024 is considered outdated)

In [11]:
def check_timeliness(df, date_col='LastUpdated', threshold_date=None):
    """Check if data is up-to-date"""
    if threshold_date is None:
        threshold_date = datetime(2024, 3, 25)

    # Convert to datetime
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    # Check which records are outdated
    outdated_mask = df[date_col] < threshold_date
    outdated_count = outdated_mask.sum()

    print("=== TIMELINESS REPORT ===")
    print(f"Threshold date: {threshold_date.strftime('%Y-%m-%d')}")
    print(f"Outdated records: {outdated_count}")
    print(f"Outdated percentage: {(outdated_count/len(df))*100:.1f}%")
    print(f"\nOutdated records:")
    print(df[outdated_mask][['Name', 'LastUpdated']])

    return {
        'threshold_date': threshold_date.strftime('%Y-%m-%d'),
        'outdated_count': int(outdated_count),
        'outdated_percentage': round((outdated_count/len(df))*100, 2)
    }

timeliness_report = check_timeliness(df)

=== TIMELINESS REPORT ===
Threshold date: 2024-03-25
Outdated records: 3
Outdated percentage: 60.0%

Outdated records:
       Name      LastUpdated
2  Johnathan Doe  2022-05-25
3     Jake Brown  2023-10-01
4     Jake Brown  2023-10-01
